# VisualGuard-LVLM — MME sensitivity experiments only (4 runs)

Fresh-runtime notebook. No Drive, no POPE, no ablations. Setup cells are copied from your working MME notebook; only the run cell is new.

Runs: `lambda=0.25`, `lambda=0.75`, `threshold=0.25`, `threshold=0.45` (LLaVA-1.5-7B, 4-bit, seed 42, full MME).

**Set Runtime -> Change runtime type -> GPU (T4) first, then run top to bottom.**

In [ ]:
# 1. GPU + download tools
import os, sys, subprocess, json, time, shutil, zipfile
from pathlib import Path

out = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(out[:2500] or 'No NVIDIA GPU detected. Switch the runtime to GPU and restart.')
assert out, 'GPU required.'
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--upgrade', 'huggingface_hub', 'requests', 'pillow', 'tqdm'], check=True)

Tue Oct 20 13:18:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', '--upgrade', 'huggingface_hub', 'requests', 'pillow', 'tqdm'], returncode=0)

In [ ]:
# 2. Clone project
REPO = Path('/content/VisualGuard-LVLM')
if not (REPO / 'src' / 'run.py').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ria0304/VisualGuard-LVLM.git', str(REPO)], check=True)
assert (REPO / 'src' / 'run.py').is_file(), f'Missing src/run.py in {REPO}'
os.chdir(REPO)
print('Project ready:', REPO)
print('Commit:', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())

Project ready: /content/VisualGuard-LVLM
Commit: b316b903cc43141f49bbafcdfefc61bcf4db6764


In [ ]:
# 3. Download MME from Hugging Face into /content
from huggingface_hub import snapshot_download

MME_SOURCE_REPO = 'yingss/mixlora-eval-data'
DOWNLOAD_ROOT = Path('/content/datasets/mme_source')
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

source_path = Path(snapshot_download(
    repo_id=MME_SOURCE_REPO, repo_type='dataset', local_dir=str(DOWNLOAD_ROOT),
    allow_patterns=['mme/**', 'README.md', '.gitattributes'], max_workers=8,
))
all_files = [p for p in source_path.rglob('*') if p.is_file()]
print('Downloaded files:', len(all_files))
if not all_files:
    raise RuntimeError('No MME files downloaded. Check internet access / the HF repo.')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2426 files:   0%|          | 0/2426 [00:00<?, ?it/s]

Downloaded files: 7281


In [ ]:
# 4. Normalize MME into <root>/<subtask>/<subtask>.jsonl + images/
MME_ROOT = Path('/content/datasets/MME')
if MME_ROOT.exists():
    shutil.rmtree(MME_ROOT)
MME_ROOT.mkdir(parents=True)

source_root = next(
    p for p in DOWNLOAD_ROOT.rglob('*')
    if p.is_dir() and p.name.lower() == 'mme_benchmark_release_version'
)
print('Source release dir:', source_root)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
YES, NO = {'yes', 'true', '1'}, {'no', 'false', '0'}

image_index = {}
for p in source_root.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMG_EXT:
        image_index.setdefault(p.name, []).append(p)
print('Images found in source:', sum(len(v) for v in image_index.values()))

def resolve_image(name, task_dir):
    cands = image_index.get(Path(name).name, [])
    for c in cands:
        if task_dir in c.parents:
            return c
    return cands[0] if cands else None

summary_rows, missing_total, total_rows = [], 0, 0
for ann in sorted(source_root.glob('*.jsonl')):
    task = ann.stem
    task_src = source_root / task
    out_dir = MME_ROOT / task
    (out_dir / 'images').mkdir(parents=True, exist_ok=True)
    rows, missing = [], []
    for line in ann.read_text(encoding='utf-8', errors='replace').splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        image = obj.get('image', obj.get('image_path', obj.get('image_name', obj.get('image_id'))))
        question = obj.get('text', obj.get('question', obj.get('prompt')))
        answer = obj.get('answer', obj.get('target', obj.get('label')))
        if image is None or question is None or answer is None:
            continue
        a = str(answer).strip().lower()
        if a not in YES | NO:
            continue
        src_img = resolve_image(str(image), task_src)
        if src_img is None:
            missing.append(str(image)); continue
        link = out_dir / 'images' / src_img.name
        if not link.exists():
            link.symlink_to(src_img)
        rows.append({'image': src_img.name, 'text': str(question), 'answer': 'Yes' if a in YES else 'No'})
    (out_dir / f'{task}.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    summary_rows.append((task, len(rows), len(missing)))
    total_rows += len(rows); missing_total += len(missing)

for t, n, m in summary_rows:
    print(f'{t:28s} {n:4d}  missing={m}')
print(f'Subtasks: {len(summary_rows)} | Rows: {total_rows} | Missing images: {missing_total}')
if len(summary_rows) != 14:
    raise RuntimeError(f'Expected 14 MME subtasks, built {len(summary_rows)}.')
if missing_total:
    raise RuntimeError('Some annotation rows point to images that were not downloaded.')

Source release dir: /content/datasets/mme_source/mme/MME_Benchmark_release_version
Images found in source: 1187
OCR                            40  missing=0
artwork                       400  missing=0
celebrity                     340  missing=0
code_reasoning                 40  missing=0
color                          60  missing=0
commonsense_reasoning         140  missing=0
count                          60  missing=0
existence                      60  missing=0
landmark                      400  missing=0
numerical_calculation          40  missing=0
position                       60  missing=0
posters                       294  missing=0
scene                         400  missing=0
text_translation               40  missing=0
Subtasks: 14 | Rows: 2374 | Missing images: 0


In [ ]:
# 5. Verify images (re-download corrupt ones, drop any that stay unreadable)
from PIL import Image
from huggingface_hub import hf_hub_download

def find_bad():
    bad = []
    for link in sorted(MME_ROOT.glob('*/images/*')):
        try:
            with Image.open(link) as im:
                im.load()
        except Exception:
            bad.append(link)
    return bad

bad = find_bad()
print(f'{len(bad)} unreadable image(s) before repair')
for b in bad:
    try:
        rel = b.resolve().relative_to(DOWNLOAD_ROOT).as_posix()
        hf_hub_download(repo_id=MME_SOURCE_REPO, repo_type='dataset', filename=rel,
                        local_dir=str(DOWNLOAD_ROOT), force_download=True)
    except Exception as e:
        print('  re-download failed for', b.name, ':', e)

bad = find_bad()
bad_names = {b.name for b in bad}
dropped = 0
if bad:
    print('Dropping corrupt image(s):', sorted(bad_names))
    for jl in MME_ROOT.glob('*/*.jsonl'):
        rows = [json.loads(l) for l in jl.read_text(encoding='utf-8').splitlines() if l.strip()]
        keep = [r for r in rows if r['image'] not in bad_names]
        if len(keep) != len(rows):
            jl.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in keep), encoding='utf-8')
            dropped += len(rows) - len(keep)
    for b in bad:
        b.unlink()
assert not find_bad(), 'Unreadable images remain.'
total = sum(1 for jl in MME_ROOT.glob('*/*.jsonl') for l in jl.read_text(encoding='utf-8').splitlines() if l.strip())
print(f'All images readable. Questions in use: {total} (excluded from 2374: {dropped})')

1 unreadable image(s) before repair


mme/MME_Benchmark_release_version/landma(…): reconstructing file:   0%|          |  0.00B / 76.9kB            

mme/MME_Benchmark_release_version/landma(…): downloading bytes:           |  0.00B            

Dropping corrupt image(s): ['03d5e3bfc958be38.jpg']
All images readable. Questions in use: 2372 (excluded from 2374: 2)


In [ ]:
# 6. Validate runtime + project loader; install 4-bit support
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'bitsandbytes>=0.46.1', 'accelerate'], check=True)

import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime > Change runtime type > T4 GPU, then rerun from the top.')

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO) + (os.pathsep + env.get('PYTHONPATH', ''))
probe = subprocess.run([sys.executable, '-m', 'src.run', '--help'], cwd=str(REPO), env=env,
                       capture_output=True, text=True, timeout=120)
HELP_TEXT = (probe.stdout or '') + (probe.stderr or '')
if probe.returncode != 0:
    print(HELP_TEXT[-3000:])
    raise RuntimeError('src.run import check failed. No model run started.')

sys.path.insert(0, str(REPO))
from src.data.mme import discover_subtasks, load_mme
subtasks = discover_subtasks(MME_ROOT)
print('Discovered subtasks:', len(subtasks))
if len(subtasks) < 14:
    raise RuntimeError(f'Expected 14 MME subtasks, found {len(subtasks)}.')
samples = load_mme(MME_ROOT)
print('Validated examples:', len(samples))
for flag in ['--lambda', '--threshold', '--run-name', '--config', '--output-dir']:
    assert flag in HELP_TEXT, f'{flag} not in src.run --help; the repo version differs from your earlier run.'
print('Dataset + CLI validation passed.')

CUDA available: True
Discovered subtasks: 14
Validated examples: 2372
Dataset + CLI validation passed.


In [ ]:
# 7. RUN THE 4 SENSITIVITY EXPERIMENTS (LLaVA-1.5-7B, 4-bit, seed 42, full MME)
MODEL = 'llava-hf/llava-1.5-7b-hf'
DEVICE = 'cuda'
DTYPE = 'float16'
QUANTIZATION = '4bit'
SEED = 42
MAX_SAMPLES = None   # keep None for a full result

RESULTS = REPO / 'results' / 'mme_only'
LOGS = RESULTS / 'logs'
RESULTS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

base_args = [
    '--benchmark', 'mme', '--model', MODEL, '--device', DEVICE,
    '--dtype', DTYPE, '--quantization', QUANTIZATION,
    '--data-root', str(MME_ROOT),
    '--seed', str(SEED), '--log-level', 'INFO',
]
if MAX_SAMPLES is not None:
    base_args += ['--max-samples', str(MAX_SAMPLES)]

EXPERIMENTS = [
    ('sensitivity_lambda_025',    'visualguard', 'visualguard', ['--lambda', '0.25']),
    ('sensitivity_lambda_075',    'visualguard', 'visualguard', ['--lambda', '0.75']),
    ('sensitivity_threshold_025', 'visualguard', 'visualguard', ['--threshold', '0.25']),
    ('sensitivity_threshold_045', 'visualguard', 'visualguard', ['--threshold', '0.45']),
]

for run_name, method, config, overrides in EXPERIMENTS:
    run_dir = RESULTS / run_name
    result_file = run_dir / f'{run_name}.json'
    log_file = LOGS / f'{run_name}.log'
    run_dir.mkdir(parents=True, exist_ok=True)

    if result_file.exists() and result_file.stat().st_size > 0:
        print(f'[SKIP] Existing result: {result_file}')
        continue

    cmd = [sys.executable, '-m', 'src.run', *base_args,
           '--output-dir', str(run_dir), '--method', method, '--config', config,
           '--run-name', run_name, *overrides]
    print(f"\n{'='*70}\nRUNNING: {run_name}\n{' '.join(cmd)}\n{'='*70}")
    t0 = time.time()
    with log_file.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log.write(line); log.flush()
        rc = proc.wait()
    print(f'Finished {run_name} | exit={rc} | {(time.time()-t0)/60:.1f} min')
    if rc != 0:
        raise RuntimeError(f'{run_name} failed. Inspect log: {log_file}')
    if not result_file.exists():
        print('WARNING: expected result file not found:', result_file, '| found:', [p.name for p in run_dir.glob('*.json')])

print('\nAll 4 sensitivity runs finished. Results under:', RESULTS)


RUNNING: sensitivity_lambda_025
/usr/bin/python3 -m src.run --benchmark mme --model llava-hf/llava-1.5-7b-hf --device cuda --dtype float16 --quantization 4bit --data-root /content/datasets/MME --seed 42 --log-level INFO --output-dir /content/VisualGuard-LVLM/results/mme_only/sensitivity_lambda_025 --method visualguard --config visualguard --run-name sensitivity_lambda_025 --lambda 0.25
2026-10-20 13:28:05,191 INFO    visualguard: Method: visualguard
2026-10-20 13:28:05,192 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-20 13:28:05,192 INFO    visualguard: Evidence weights: alpha=1.0 beta=1.0 gamma=0.0 lambda=0.25 threshold=0.35
2026-10-20 13:28:05,192 INFO    visualguard: MME subtasks discovered: ['existence', 'count', 'position', 'color', 'posters', 'celebrity', 'scene', 'landmark', 'artwork', 'OCR', 'commonsense_reasoning', 'numerical_calculation', 'text_translation', 'code_reasoning']
2026-10-20 13:28:13,905 INFO    numexpr.utils: NumExpr defaulting to 2 threads.
2026-

In [ ]:
# 8. Summary table of the 4 runs
import pandas as pd
rows = []
for run_name, *_ in EXPERIMENTS:
    for p in (RESULTS / run_name).glob('*.json'):
        try:
            d = json.loads(p.read_text(encoding='utf-8'))
        except Exception:
            continue
        m = d.get('mme', {})
        ct = m.get('category_totals', {})
        eff = d.get('efficiency', {})
        rows.append({'run': run_name, 'MME_total': ct.get('total_score'),
                     'perception': ct.get('perception_score'), 'cognition': ct.get('cognition_score'),
                     'n_samples': m.get('n_samples'),
                     'runtime_min': round(eff.get('total_runtime_s', 0) / 60, 2),
                     'peak_gpu_mb': eff.get('peak_gpu_memory_mb')})
df = pd.DataFrame(rows)
print(df.to_string(index=False) if not df.empty else 'No result JSONs found.')
if not df.empty:
    df.to_csv(RESULTS / 'sensitivity_summary.csv', index=False)

                      run   MME_total  perception  cognition  n_samples  runtime_min  peak_gpu_mb
   sensitivity_lambda_025 1776.511434 1466.154291 310.357143       2372        76.07 11401.773438
   sensitivity_lambda_075 1774.510177 1464.153035 310.357143       2372        73.36 11401.773438
sensitivity_threshold_025 1776.379081 1466.021938 310.357143       2372        73.73 11401.773438
sensitivity_threshold_045 1770.469361 1460.112218 310.357143       2372        73.79 11401.773438


In [10]:
# 9. Zip and download (Colab storage is temporary)
from google.colab import files
zip_base = '/content/VisualGuard_MME_Sensitivity_Results'
shutil.make_archive(zip_base, 'zip', root_dir=RESULTS)
files.download(zip_base + '.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>